**PROYECTO**       : GESTION INTEGRAL PASIVOS  
**NOMBRE**         : nb_fct_saldo_cuenta_pasivo.ipynb  
**TABLA DESTINO**  : mb_gold_prod.pasivos.fct_saldo_cuenta_pasivo  
**TABLA FUENTE**   : mb_silver_prod.mmff.h_saldo_cuenta  
**OBJETIVO**       : Poblar el hecho de saldos de cuentas pasivas  
**TIPO**           : PYTHON  
**REPROCESABLE**   : SI  
**OBSERVACION**    : NA  
**SCHEDULER**      : NA  
**JOB**            : NA
| VERSION | DESARROLLADOR | PROVEEDOR | PO | FECHA | DESCRIPCION |
|---------|---------------|-----------|----|-------|-------------|
| 1.0 | Kevin Ponce | MIBANCO | Enith Rodriguez | 2026-09-11 | Creacion de proceso |

## 1. Librerias y dependencias

In [0]:
import logging
import time

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Funciones de transformacion

In [0]:
def read_saldos(tabla, columnas, fecha_proceso, estado):
    """Lee los saldos proyectando y filtrando desde el origen.

    Los filtros se aplican en la lectura para no arrastrar registros que
    luego se descartan en el cruce.
    """
    return (
        spark.table(tabla)
        .select(*columnas)
        .filter(F.col("fec_proceso") == fecha_proceso)
        .filter(F.col("est_cuenta") == estado)
    )


def add_saldo_promedio(df_saldos):
    """Calcula el saldo promedio por cuenta."""
    ventana = Window.partitionBy("cod_cuenta")
    return df_saldos.withColumn(
        "mto_saldo_promedio", F.avg(F.col("mto_saldo")).over(ventana)
    )

## 3. Parametros de entrada

In [0]:
dbutils.widgets.text("ambiente", "dev")
dbutils.widgets.text("fechaproceso", "")

var_ambiente = dbutils.widgets.get("ambiente")
var_fechaproceso = dbutils.widgets.get("fechaproceso")

logger = logging.getLogger("FCT_SALDO_CUENTA_PASIVO")
logger.setLevel(logging.INFO)
ini_proceso = time.perf_counter()
logger.info("Inicio del proceso. ambiente=%s fechaproceso=%s",
            var_ambiente, var_fechaproceso)

## 4. Constantes y variables

In [0]:
TBL_SALDO_SRC = f"mb_silver_{var_ambiente}.mmff.h_saldo_cuenta"
TBL_AGENCIA_SRC = f"mb_silver_{var_ambiente}.mmff.m_agencia"
TBL_SALDO_FIN = f"mb_gold_{var_ambiente}.pasivos.fct_saldo_cuenta_pasivo"

COLUMNAS_ORIGEN = [
    "cod_cuenta",
    "cod_agencia",
    "mto_saldo",
    "est_cuenta",
    "fec_proceso",
]

EST_VIGENTE = "VIGENTE"

## 5. Logica principal

In [0]:
ini_etapa = time.perf_counter()

df_saldos = read_saldos(
    TBL_SALDO_SRC, COLUMNAS_ORIGEN, var_fechaproceso, EST_VIGENTE
)
df_agencias = spark.table(TBL_AGENCIA_SRC).select("cod_agencia", "nom_agencia")

df_saldos_enriquecido = add_saldo_promedio(
    df_saldos.join(df_agencias, "cod_agencia", "left")
)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 6. Escritura

In [0]:
try:
    (
        df_saldos_enriquecido
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_SALDO_FIN)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_SALDO_FIN, exc)
    raise

logger.info("Fin del proceso. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)